In [1]:
import vitaldb
import numpy as np
import pandas as pd

from tqdm.auto import tqdm
from scipy.signal import find_peaks, resample_poly

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def extract_bp_from_abp(abp_window, fs=500):

    # -----------------------------------------
    # Remove NaNs before processing
    # -----------------------------------------

    if np.isnan(abp_window).any():
        return None, None

    # -----------------------------------------
    # Detect systolic peaks
    # -----------------------------------------

    peaks, _ = find_peaks(
        abp_window,
        distance=int(0.4 * fs),
        prominence=5
    )

    # Need at least 2 beats
    if len(peaks) < 2:
        return None, None

    # -----------------------------------------
    # SBP
    # -----------------------------------------

    sbp_values = abp_window[peaks]

    # -----------------------------------------
    # DBP
    # -----------------------------------------

    dbp_values = []

    for i in range(len(peaks) - 1):

        beat_segment = abp_window[
            peaks[i]:peaks[i + 1]
        ]

        if len(beat_segment) > 0:

            dbp_values.append(
                np.min(beat_segment)
            )

    if len(dbp_values) == 0:
        return None, None

    # Median across beats
    sbp = np.median(sbp_values)
    dbp = np.median(dbp_values)

    # -----------------------------------------
    # Sanity checks
    # -----------------------------------------

    if sbp < 50 or sbp > 250:
        return None, None

    if dbp < 20 or dbp > 150:
        return None, None

    return sbp, dbp